In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import os

# Determinism for stability (score-neutral; keeps runs comparable while we tune only calibration)
os.environ["PYTHONHASHSEED"] = "0"
np.random.seed(0)



In [2]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split



In [3]:
from sklearn.neural_network import MLPClassifier



In [4]:
from pylab import rcParams

rcParams["figure.figsize"] = (10, 10)



In [5]:
import os


def _resolve_input_path(filename):
    candidates = [
        "../input/" + filename,
        "/kaggle/input/leaf-classification/" + filename,
        "/kaggle/input/" + filename,
        "/kaggle/data/leaf-classification/" + filename,
        "/kaggle/data/" + filename,
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[0]


train_path = _resolve_input_path("train.csv")
test_path = _resolve_input_path("test.csv")
sample_path = _resolve_input_path("sample_submission.csv")

train_path, test_path, sample_path



('../input/train.csv', '../input/test.csv', '../input/sample_submission.csv')

In [6]:
data = pd.read_csv(train_path)
parent_data = data.copy()  # keep a copy of original data
ID = data.pop("id")
data.shape



(891, 193)

In [7]:
y = data.pop("species")
le = LabelEncoder()
y = le.fit_transform(y)
print(y.shape)



(891,)


In [8]:
scaler = StandardScaler()
X = scaler.fit_transform(data.values)
print(X.shape)



(891, 192)


In [9]:
mlp = MLPClassifier(
    hidden_layer_sizes=(1024, 512),
    activation="relu",
    solver="adam",
    alpha=2e-6,  # keep identical: training core unchanged
    batch_size=192,
    learning_rate_init=0.001,
    max_iter=800,  # keep identical
    tol=1e-4,
    shuffle=True,
    random_state=0,
    early_stopping=False,
    n_iter_no_change=300,  # keep identical
    verbose=False,
)



In [10]:
mlp.fit(X, y)



MLPClassifier(alpha=2e-06, batch_size=192, hidden_layer_sizes=(1024, 512),
              max_iter=800, n_iter_no_change=300, random_state=0)

In [11]:
if hasattr(mlp, "loss_"):
    print("Final training loss:", mlp.loss_)



Final training loss: 4.21536941816114e-05


In [12]:
test = pd.read_csv(test_path)
index = test.pop("id").values

X_test = scaler.transform(test.values)



In [13]:
yPred = mlp.predict_proba(X_test)



In [14]:
sample = pd.read_csv(sample_path)
class_cols = [c for c in sample.columns if c != "id"]

pred_df = pd.DataFrame(yPred, columns=le.classes_)
pred_df = pred_df.reindex(columns=class_cols, fill_value=0.0)

pred_vals = pred_df.to_numpy(dtype=np.float64)

# Change (score-driven, minimal): reduce softening toward a near no-op.
# T=1.06 likely over-smooths probabilities and can worsen logloss by underweighting the true class.
# Moving to T=1.02 keeps the same semantics but typically improves multiclass logloss when too-uniform.
temperature = 1.02
if temperature != 1.0:
    pred_vals = np.power(pred_vals, 1.0 / temperature)

# Keep uniform mixing off (unchanged) to avoid blurring correct confident predictions.
alpha_uniform = 0.0
if alpha_uniform != 0.0:
    n_classes = pred_vals.shape[1]
    pred_vals = (1.0 - alpha_uniform) * pred_vals + alpha_uniform * (1.0 / n_classes)

# Keep Kaggle-consistent clipping and explicit row-normalization (unchanged core evaluation semantics).
eps = 1e-15  # Kaggle clips at 1e-15 anyway
pred_vals = np.clip(pred_vals, eps, 1.0 - eps)

row_sums = pred_vals.sum(axis=1, keepdims=True)
row_sums[row_sums == 0.0] = 1.0
pred_vals = pred_vals / row_sums

pred_df = pd.DataFrame(pred_vals, columns=pred_df.columns)

submission = pd.DataFrame({"id": index})
submission = pd.concat([submission, pred_df], axis=1)

submission[class_cols] = submission[class_cols].clip(0.0, 1.0)

submission.head()



,id,Acer_Capillipes,Acer_Circinatum,Acer_Mono,Acer_Opalus,Acer_Palmatum,Acer_Pictum,Acer_Platanoids,Acer_Rubrum,Acer_Rufinerve,...,Salix_Fragilis,Salix_Intergra,Sorbus_Aria,Tilia_Oliveri,Tilia_Platyphyllos,Tilia_Tomentosa,Ulmus_Bergmanniana,Viburnum_Tinus,Viburnum_x_Rhytidophylloides,Zelkova_Serrata
0,1202,1.803571e-09,7.621254e-08,1.291299e-06,1.718337e-07,1.428794e-07,1.908745e-06,3.198267e-07,7.072131e-08,1.218660e-09,...,2.012288e-08,3.673135e-08,3.174665e-09,7.166908e-09,2.345936e-09,2.187075e-07,2.646412e-09,1.110544e-09,1.054585e-06,5.931590e-08
1,992,3.887777e-07,6.195751e-09,2.260043e-11,1.758768e-09,2.053441e-09,2.748411e-08,7.080665e-10,9.307282e-10,5.484107e-12,...,5.378501e-08,2.852699e-09,9.215027e-02,7.893080e-09,1.414397e-06,3.934674e-13,1.173489e-05,2.698128e-10,7.052287e-04,1.074789e-08
2,491,1.950570e-10,9.316219e-10,2.458870e-10,5.455024e-10,9.279451e-08,1.766493e-07,4.641881e-11,5.944161e-10,2.046296e-11,...,1.282549e-10,4.796701e-11,1.970829e-10,6.768204e-12,1.458463e-09,5.928943e-09,2.037434e-11,6.442093e-13,2.027886e-06,1.144674e-09
3,1201,2.229498e-10,2.024566e-10,1.713224e-06,8.755786e-07,2.887349e-11,6.172236e-12,2.737592e-08,2.667235e-11,1.339016e-12,...,1.697631e-07,9.978732e-01,3.659579e-11,7.137654e-12,1.833624e-12,9.153021e-10,8.265587e-12,1.959923e-05,1.773227e-10,3.895338e-09
4,72,1.400568e-12,1.054407e-11,1.017611e-08,4.854266e-10,9.999933e-16,5.756008e-10,1.830606e-10,6.859157e-15,2.555544e-15,...,3.574887e-09,8.030942e-12,1.296915e-14,9.999933e-16,9.999933e-16,2.591575e-11,3.246343e-12,3.614241e-09,5.259115e-13,5.855959e-14


In [15]:
out_path = "submission_nn_kernel.csv"
submission.to_csv(out_path, index=False)
print("Wrote:", out_path, "shape:", submission.shape)
print(submission.columns[:5].tolist(), "...", submission.columns[-5:].tolist())

Wrote: submission_nn_kernel.csv shape: (99, 100)
['id', 'Acer_Capillipes', 'Acer_Circinatum', 'Acer_Mono', 'Acer_Opalus'] ... ['Tilia_Tomentosa', 'Ulmus_Bergmanniana', 'Viburnum_Tinus', 'Viburnum_x_Rhytidophylloides', 'Zelkova_Serrata']
